In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Tweets.csv')
df.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [3]:
df.shape

(27481, 4)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27481 entries, 0 to 27480
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   textID         27481 non-null  str  
 1   text           27480 non-null  str  
 2   selected_text  27480 non-null  str  
 3   sentiment      27481 non-null  str  
dtypes: str(4)
memory usage: 858.9 KB


In [5]:
df.describe()

,textID,text,selected_text,sentiment
count,27481,27480,27480,27481
unique,27481,27480,22463,3
top,cb774db0d1,"I`d have responded, if I were going",good,neutral
freq,1,1,199,11118


In [6]:
df.isna().sum()

textID           0
text             1
selected_text    1
sentiment        0
dtype: int64

In [7]:
df.dropna(inplace=True)

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.columns

Index(['textID', 'text', 'selected_text', 'sentiment'], dtype='str')

In [10]:
df.drop(columns=['selected_text'], inplace=True)
df.head()

,textID,text,sentiment
0,cb774db0d1,"I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,negative
2,088c60f138,my boss is bullying me...,negative
3,9642c003ef,what interview! leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...",negative


In [11]:
df['sentiment'].value_counts()

sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64

In [12]:
df['text'] = df['text'].str.lower().str.strip()
df.head(10)

,textID,text,sentiment
0,cb774db0d1,"i`d have responded, if i were going",neutral
1,549e992a42,sooo sad i will miss you here in san diego!!!,negative
2,088c60f138,my boss is bullying me...,negative
3,9642c003ef,what interview! leave me alone,negative
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",negative
5,28b57f3990,http://www.dothebouncy.com/smf - some shameles...,neutral
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,positive
7,50e14c0bb8,soooo high,neutral
8,e050245fbd,both of you,neutral
9,fc2cbefa9d,journey!? wow... u just became cooler. hehe.....,positive


In [ ]:
df['sentiment'] = df['sentiment'].map({
    'neutral': 0,
    'positive': 1,
    'negative': -1
})
df.head()
# mapping for easy workflow

,textID,text,sentiment
0,cb774db0d1,"i`d have responded, if i were going",0
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1
2,088c60f138,my boss is bullying me...,-1
3,9642c003ef,what interview! leave me alone,-1
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# downloading required resources for text preprocessing
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
stop_words = set(stopwords.words("english"))

contradictions = {
    "don't": "do not",
    "didn't": "did not",
    "isn't": "is not",
    "wasn't": "was not",
    "couldn't": "could not",
    "won't": "will not",
    "can't": "cannot",
    "wouldn't": "would not",
    "shouldn't": "should not"
}

negation_words = {"no", "not", "nor", "never"}
custom_stopwords = stop_words - negation_words
# taking care of negative words too.

In [ ]:
import re
# regex for preprocessing hidden problems
def text_preprocess(text):
    text = str(text).lower()
    for contradiction, expand in contradictions.items():
        text = re.sub(rf"\b{re.escape(contradiction)}\b",expand, text)
    
    tokenized_word = word_tokenize(text)
    return ' '.join(
        word for word in tokenized_word
        if word.lower() not in custom_stopwords)

In [17]:
df.head()

,textID,text,sentiment
0,cb774db0d1,"i`d have responded, if i were going",0
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1
2,088c60f138,my boss is bullying me...,-1
3,9642c003ef,what interview! leave me alone,-1
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # removing unecessary links
    return text

In [ ]:
def remove_punctuation(text):
    return re.sub(r"""[!,`.%$#@?;:_+~()\[\]/\\-]""", '', str(text)) # removing special characters

In [20]:
df['stemmed'] = df['text'].fillna('').apply(clean_text)
df['stemmed'] = df['stemmed'].apply(text_preprocess)
df['stemmed'] = df['stemmed'].apply(remove_punctuation)

In [21]:
df.head()

,textID,text,sentiment,stemmed
0,cb774db0d1,"i`d have responded, if i were going",0,responded going
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,-1,boss bullying
3,9642c003ef,what interview! leave me alone,-1,interview leave alone
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1,sons * * * * put releases already bought


In [22]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming(text):
    tokenized = word_tokenize(text)
    return ' '.join(ps.stem(word) for word in tokenized)

df['stemmed'] = df['stemmed'].apply(stemming)
df.head()

,textID,text,sentiment,stemmed
0,cb774db0d1,"i`d have responded, if i were going",0,respond go
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,-1,boss bulli
3,9642c003ef,what interview! leave me alone,-1,interview leav alon
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1,son * * * * put releas alreadi bought


In [23]:
df.head(10)

,textID,text,sentiment,stemmed
0,cb774db0d1,"i`d have responded, if i were going",0,respond go
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,-1,boss bulli
3,9642c003ef,what interview! leave me alone,-1,interview leav alon
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1,son * * * * put releas alreadi bought
5,28b57f3990,http://www.dothebouncy.com/smf - some shameles...,0,shameless plug best ranger forum earth
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,1,2am feed babi fun smile coo
7,50e14c0bb8,soooo high,0,soooo high
8,e050245fbd,both of you,0,
9,fc2cbefa9d,journey!? wow... u just became cooler. hehe.....,1,journey wow u becam cooler hehe possibl


In [24]:
from sklearn.model_selection import train_test_split

X = df['stemmed']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_stem = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_idf = tfidf_stem.fit_transform(X_train)
X_test_idf = tfidf_stem.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
# using multi not gaussian because we have more than 2 classes!
nb = MultinomialNB()
nb.fit(X_train_idf, y_train)
y_pred = nb.predict(X_test_idf)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
# not good performance

Accuracy: 0.6381004366812227

Classification Report:
              precision    recall  f1-score   support

          -1       0.74      0.48      0.58      1556
           0       0.55      0.78      0.65      2223
           1       0.77      0.59      0.67      1717

    accuracy                           0.64      5496
   macro avg       0.69      0.62      0.63      5496
weighted avg       0.67      0.64      0.64      5496


Confusion Matrix:
[[ 751  749   56]
 [ 228 1745  250]
 [  37  669 1011]]


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# downloading lemmatizer resources
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\abc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [29]:
def lemmatize(text):
    tokenized = word_tokenize(str(text))
    lemmatized = [lemmatizer.lemmatize(word) for word in tokenized]

    return ' '.join(lemmatized)

In [30]:
df.columns

Index(['textID', 'text', 'sentiment', 'stemmed'], dtype='str')

In [31]:
df['lemmatized'] = df['text'].fillna('').apply(clean_text)
df['lemmatized'] = df['lemmatized'].apply(text_preprocess)
df['lemmatized'] = df['lemmatized'].fillna('').apply(remove_punctuation)
df.head()

,textID,text,sentiment,stemmed,lemmatized
0,cb774db0d1,"i`d have responded, if i were going",0,respond go,responded going
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1,sooo sad miss san diego,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,-1,boss bulli,boss bullying
3,9642c003ef,what interview! leave me alone,-1,interview leav alon,interview leave alone
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1,son * * * * put releas alreadi bought,sons * * * * put releases already bought


In [32]:
df['lemmatized'] = df['lemmatized'].apply(lemmatize)
df.head()

,textID,text,sentiment,stemmed,lemmatized
0,cb774db0d1,"i`d have responded, if i were going",0,respond go,responded going
1,549e992a42,sooo sad i will miss you here in san diego!!!,-1,sooo sad miss san diego,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,-1,boss bulli,bos bullying
3,9642c003ef,what interview! leave me alone,-1,interview leav alon,interview leave alone
4,358bd9e861,"sons of ****, why couldn`t they put them on th...",-1,son * * * * put releas alreadi bought,son * * * * put release already bought


In [33]:
X1 = df['lemmatized']
y1 = df['sentiment']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y)

In [34]:
tfidf_lemma = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X1_train_idf = tfidf_lemma.fit_transform(X1_train)
X1_test_idf = tfidf_lemma.transform(X1_test)

In [35]:
nb1 = MultinomialNB()
nb1.fit(X1_train_idf, y1_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3,)","[6225.,8894.,6865.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3,)","[-1.26,-0.9 ,-1.16]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](3,)","[-1, 0, 1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3, 10000)","[[0.38,0.82,0.41,...,1.7 ,0.73,1.01], [0.79,0. ,0.78,...,2.87,0.38,1.93], [0. ,0. ,0.4 ,...,0.27,0. ,0.32]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3, 10000)","[[ -9.84, -9.56, -9.82,..., -9.17, -9.61, -9.46], [ -9.76,-10.34, -9.77,..., -8.99,-10.02, -9.27], [-10.23,-10.23, -9.9 ,...,-10. ,-10.23, -9.95]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10000


In [36]:
y_predict1 = nb1.predict(X1_test_idf)

In [ ]:
print("Accuracy:", accuracy_score(y1_test, y_predict1))

print("\nClassification Report:")
print(classification_report(y1_test, y_predict1))

print("\nConfusion Matrix:")
print(confusion_matrix(y1_test, y_predict1))
# not good performance even with lemmatization for NB

Accuracy: 0.6370087336244541

Classification Report:
              precision    recall  f1-score   support

          -1       0.73      0.48      0.58      1556
           0       0.55      0.78      0.65      2223
           1       0.77      0.59      0.67      1717

    accuracy                           0.64      5496
   macro avg       0.68      0.62      0.63      5496
weighted avg       0.67      0.64      0.63      5496


Confusion Matrix:
[[ 746  749   61]
 [ 231 1743  249]
 [  46  659 1012]]


## Stemming Logistic

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_stem = LogisticRegression(max_iter=1000, random_state=42)

In [39]:
logistic_stem.fit(X_train_idf, y_train)
y_pred_logistic_stem = logistic_stem.predict(X_test_idf)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_logistic_stem))

print("\nMacro F1 Score:",
      f1_score(y_test, y_pred_logistic_stem, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic_stem))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic_stem))
# improved performance

Accuracy: 0.6888646288209607

Macro F1 Score: 0.6909067100901649

Classification Report:
              precision    recall  f1-score   support

          -1       0.73      0.62      0.67      1556
           0       0.62      0.73      0.67      2223
           1       0.77      0.69      0.73      1717

    accuracy                           0.69      5496
   macro avg       0.71      0.68      0.69      5496
weighted avg       0.70      0.69      0.69      5496


Confusion Matrix:
[[ 969  526   61]
 [ 308 1631  284]
 [  59  472 1186]]


## Lemma Logistic

In [41]:
logistic_lemma = LogisticRegression(max_iter=1000, random_state=42)
logistic_lemma.fit(X1_train_idf, y1_train)
y1_pred_logistic_lema = logistic_lemma.predict(X1_test_idf)

In [ ]:
print("Accuracy:", accuracy_score(y1_test, y1_pred_logistic_lema))

print("\nMacro F1 Score:",
      f1_score(y1_test, y1_pred_logistic_lema, average="macro"))

print("\nClassification Report:")
print(classification_report(y1_test, y1_pred_logistic_lema))

print("\nConfusion Matrix:")
print(confusion_matrix(y1_test, y1_pred_logistic_lema))
# stemming logistic was better

Accuracy: 0.6843158660844251

Macro F1 Score: 0.686055110695711

Classification Report:
              precision    recall  f1-score   support

          -1       0.71      0.62      0.66      1556
           0       0.62      0.73      0.67      2223
           1       0.78      0.69      0.73      1717

    accuracy                           0.68      5496
   macro avg       0.70      0.68      0.69      5496
weighted avg       0.69      0.68      0.69      5496


Confusion Matrix:
[[ 957  539   60]
 [ 316 1627  280]
 [  66  474 1177]]


### Testing

In [43]:
texts = [
    "This game is absolutely amazing, I love every minute of it!",
    "The product is terrible and a complete waste of money.",
    "The package arrived yesterday.",
    "The graphics are beautiful, but the gameplay is boring.",
    "I thought this movie would be bad, but it was surprisingly good."
]
example = pd.DataFrame(columns=['Text'])
example['Text'] = texts

In [44]:
example

,Text
0,"This game is absolutely amazing, I love every ..."
1,The product is terrible and a complete waste o...
2,The package arrived yesterday.
3,"The graphics are beautiful, but the gameplay i..."
4,"I thought this movie would be bad, but it was ..."


In [45]:
example['stemmed'] = example['Text'].fillna('').apply(clean_text)
example['stemmed'] = example['stemmed'].apply(text_preprocess)
example['stemmed'] = example['stemmed'].apply(remove_punctuation)
example['stemmed'] = example['stemmed'].apply(stemming)
example.head()

,Text,stemmed
0,"This game is absolutely amazing, I love every ...",game absolut amaz love everi minut
1,The product is terrible and a complete waste o...,product terribl complet wast money
2,The package arrived yesterday.,packag arriv yesterday
3,"The graphics are beautiful, but the gameplay i...",graphic beauti gameplay bore
4,"I thought this movie would be bad, but it was ...",thought movi would bad surprisingli good


In [ ]:
X_example = tfidf_stem.transform(example['stemmed'])
predictions = logistic_stem.predict(X_example)
for text, prediction in zip(example['Text'], predictions):
    print(text, "->", prediction)
    # neutral statements even though were in big amount. But still missed one in testing.

This game is absolutely amazing, I love every minute of it! -> 1
The product is terrible and a complete waste of money. -> -1
The package arrived yesterday. -> -1
The graphics are beautiful, but the gameplay is boring. -> -1
I thought this movie would be bad, but it was surprisingly good. -> 1


In [47]:
text = "The package arrived yesterday."

text = clean_text(text)
text = text_preprocess(text)
text = remove_punctuation(text)
text = stemming(text)

X_new = tfidf_stem.transform([text])
print("Processed text:", text)
print("Predicted sentiment:", logistic_stem.predict(X_new)[0])
print("Class order:", logistic_stem.classes_)
print("Probabilities:", logistic_stem.predict_proba(X_new))

Processed text: packag arriv yesterday
Predicted sentiment: -1
Class order: [-1  0  1]
Probabilities: [[0.44792682 0.260959   0.29111418]]


In [48]:
df['sentiment'].value_counts()

sentiment
 0    11117
 1     8582
-1     7781
Name: count, dtype: int64

In [49]:
model_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)

model_balanced.fit(X_train_idf, y_train)
y_pred_balanced = model_balanced.predict(X_test_idf)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nMacro F1 Score:",
      f1_score(y_test, y_pred_balanced, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))
#performance improved even more

Accuracy: 0.6994177583697234

Macro F1 Score: 0.7032807124030546

Classification Report:
              precision    recall  f1-score   support

          -1       0.68      0.72      0.70      1556
           0       0.67      0.65      0.66      2223
           1       0.76      0.74      0.75      1717

    accuracy                           0.70      5496
   macro avg       0.70      0.70      0.70      5496
weighted avg       0.70      0.70      0.70      5496


Confusion Matrix:
[[1119  365   72]
 [ 441 1454  328]
 [  86  360 1271]]


In [51]:
y_pred_example = model_balanced.predict(X_example)

for text, prediction in zip(example['Text'], y_pred_example):
    print(text, "->", prediction)

This game is absolutely amazing, I love every minute of it! -> 1
The product is terrible and a complete waste of money. -> -1
The package arrived yesterday. -> -1
The graphics are beautiful, but the gameplay is boring. -> -1
I thought this movie would be bad, but it was surprisingly good. -> 1


In [ ]:
# still same problem. Model needs to be tuned for neutral and change weightage values